# 🚨 Deep Learning for Comment Toxicity Detection
### Multi-Label Text Classification using Bidirectional LSTM — **PyTorch Version**

**Domain:** Online Community Management and Content Moderation  
**Dataset:** train.csv — 159,571 comments with 6 toxicity labels  
**Framework:** PyTorch (replaces TensorFlow/Keras)  
**Model:** Bidirectional LSTM (BiLSTM)  
**Labels:** toxic, severe_toxic, obscene, threat, insult, identity_hate  

---
### 🔄 What Changed from the TensorFlow Version?

| Step | TensorFlow / Keras | PyTorch (this notebook) |
|---|---|---|
| Import | `import tensorflow as tf` | `import torch`, `torch.nn` |
| Tokenizer | `Keras Tokenizer` | Manual vocabulary with `collections.Counter` |
| Padding | `pad_sequences()` | Manual `encode_and_pad()` function |
| Model | `keras.Sequential([...])` | `class BiLSTMClassifier(nn.Module)` |
| Training | `model.fit()` one line | Manual `for epoch / for batch` loop |
| Save | `model.save('model.h5')` | `torch.save(model.state_dict(), path)` |
| Inference | `model.predict(array)` | `model(tensor)` inside `torch.no_grad()` |
| Data Pipeline | Raw NumPy arrays | `Dataset` + `DataLoader` classes |


## Step 1: Install and Import All Required Libraries


In [ ]:
# Install required libraries (uncomment and run once if needed)
# !pip install torch pandas numpy scikit-learn matplotlib seaborn streamlit

# ── Standard Python libraries ─────────────────────────────────────────────
import os
import re
import string
import pickle
import warnings
import collections    # used to build vocabulary by counting word frequencies

# ── Data handling ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine learning utilities ────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             hamming_loss, roc_auc_score)

# ── PyTorch — replaces ALL of tensorflow.keras ───────────────────────────
import torch
import torch.nn as nn                          # neural network layers
import torch.optim as optim                    # optimizers (Adam etc.)
from torch.utils.data import Dataset, DataLoader  # data pipeline

warnings.filterwarnings('ignore')

# Fix random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# PyTorch uses DEVICE to know whether to run on GPU or CPU
# In TensorFlow this is handled automatically — in PyTorch we set it explicitly
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('PyTorch version :', torch.__version__)
print('Device in use   :', DEVICE)   # 'cuda' if GPU available, otherwise 'cpu'
print('All libraries imported successfully!')

## Constants — Key Parameters You Can Change


In [ ]:
# ── Vocabulary and Sequence settings ────────────────────────────────────
MAX_VOCAB    = 30000   # Keep the 30,000 most common words in vocabulary
MAX_SEQ_LEN  = 200     # Pad or trim every comment to exactly 200 words
EMBEDDING_DIM = 64     # Each word is represented by a 64-number vector
HIDDEN_DIM   = 64      # Number of hidden units in the LSTM (per direction)

# ── Training settings ───────────────────────────────────────────────────
BATCH_SIZE   = 128
EPOCHS       = 5
VAL_SPLIT    = 0.2     # Use 20% of training data for validation
THRESHOLD    = 0.5     # Probability >= 0.5 is flagged as toxic

# ── Label names ─────────────────────────────────────────────────────────
LABEL_COLS   = ['toxic', 'severe_toxic', 'obscene',
                'threat', 'insult', 'identity_hate']

# ── File paths ───────────────────────────────────────────────────────────
MODEL_PATH   = 'toxicity_model_pytorch.pth'  # .pth is the PyTorch convention
VOCAB_PATH   = 'vocab_pytorch.pkl'
CONFIG_PATH  = 'model_config_pytorch.pkl'
OUTPUT_CSV   = 'test_predictions_pytorch.csv'

# Special vocabulary tokens
PAD_TOKEN = '<PAD>'   # Padding token — fills short comments to MAX_SEQ_LEN
OOV_TOKEN = '<OOV>'   # Out-Of-Vocabulary — replaces unknown words at inference

print('Constants set!')
print(f'Vocabulary size  : {MAX_VOCAB:,}')
print(f'Sequence length  : {MAX_SEQ_LEN}')
print(f'Labels           : {LABEL_COLS}')

## Step 2: Load the Dataset


In [ ]:
# Load training and test CSV files
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print('Train shape :', train_df.shape)   # (159571, 8)
print('Test shape  :', test_df.shape)    # (153164, 2)

# Show the first 5 rows
train_df.head()

In [ ]:
# Check columns
print('Train columns :', train_df.columns.tolist())
print('Test columns  :', test_df.columns.tolist())

# Output:
# Train: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene',
#         'threat', 'insult', 'identity_hate']
# Test:  ['id', 'comment_text']

## Step 3: Exploratory Data Analysis (EDA)

This step is **identical in both TensorFlow and PyTorch** — EDA is framework-agnostic.


In [ ]:
# Check for missing values and duplicates
print('Missing values :', train_df.isnull().sum().sum())  # 0
print('Duplicate rows :', train_df.duplicated().sum())    # 0

# Count how many comments have each label
label_counts = train_df[LABEL_COLS].sum().sort_values(ascending=False)
print('\nComments per toxicity label:')
print(label_counts)

In [ ]:
# How many comments are clean vs toxic?
clean_n = (train_df[LABEL_COLS].sum(axis=1) == 0).sum()
toxic_n = (train_df[LABEL_COLS].sum(axis=1) >  0).sum()

print(f'Clean comments : {clean_n:,}  ({clean_n/len(train_df)*100:.1f}%)')
print(f'Toxic comments : {toxic_n:,}  ({toxic_n/len(train_df)*100:.1f}%)')

# Output:
# Clean comments : 143,346  (89.8%)
# Toxic comments :  16,225  (10.2%)

In [ ]:
# Plot 1 — Label distribution
plt.figure(figsize=(10, 5))
label_counts.plot(kind='bar', color='tomato', edgecolor='black', alpha=0.85)
plt.title('Number of Comments per Toxicity Label', fontsize=14, fontweight='bold')
plt.xlabel('Toxicity Label')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150)
plt.show()
print('Saved: label_distribution.png')

In [ ]:
# Plot 2 — Class imbalance
plt.figure(figsize=(6, 5))
bars = plt.bar(['Clean Comments', 'Toxic Comments'],
               [clean_n, toxic_n],
               color=['steelblue', 'tomato'], edgecolor='black')
plt.title('Class Imbalance: Clean vs Toxic', fontsize=13, fontweight='bold')
plt.ylabel('Number of Comments')
for bar, val in zip(bars, [clean_n, toxic_n]):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 500,
             f'{val:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('class_imbalance.png', dpi=150)
plt.show()

In [ ]:
# Plot 3 — Comment length distribution
train_df['comment_length'] = train_df['comment_text'].str.len()
mean_len = train_df['comment_length'].mean()

plt.figure(figsize=(10, 4))
plt.hist(train_df['comment_length'], bins=50,
         color='steelblue', edgecolor='black', alpha=0.8)
plt.axvline(mean_len, color='red', linestyle='--',
            label=f'Mean: {mean_len:.0f} chars')
plt.title('Distribution of Comment Lengths', fontsize=13, fontweight='bold')
plt.xlabel('Number of Characters')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('comment_lengths.png', dpi=150)
plt.show()

print(f'Average comment length : {mean_len:.0f} characters')
print(f'Max comment length     : {train_df["comment_length"].max()} characters')

In [ ]:
# Plot 4 — Label co-occurrence heatmap
# Shows which toxic labels tend to appear together in the same comment
plt.figure(figsize=(8, 6))
cooccur = train_df[LABEL_COLS].T.dot(train_df[LABEL_COLS])
sns.heatmap(cooccur, annot=True, fmt='d', cmap='Reds', linewidths=0.5)
plt.title('Label Co-occurrence Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('label_cooccurrence.png', dpi=150)
plt.show()

# Key finding: 'toxic' and 'insult' very often appear together

## Step 4: Text Preprocessing

This step is also **identical in both versions** — text cleaning does not depend on the framework.


In [ ]:
def clean_text(text):
    """
    Cleans a raw comment string in 5 steps:
    1. Lowercase everything
    2. Remove newlines and tabs
    3. Remove URLs
    4. Remove punctuation
    5. Collapse extra whitespace
    """
    text = text.lower()
    text = text.replace('\n', ' ').replace('\t', ' ')
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Test the function
sample = "Check this out!! Visit http://example.com\nIt's AMAZING!"
print('Before:', sample)
print('After :', clean_text(sample))

In [ ]:
# Apply to all comments
print('Cleaning training comments...')
train_df['clean_text'] = train_df['comment_text'].apply(clean_text)

print('Cleaning test comments...')
test_df['clean_text']  = test_df['comment_text'].apply(clean_text)

print('Done!')

# Quick sanity check
print('\nOriginal :', train_df['comment_text'].iloc[1][:120])
print('Cleaned  :', train_df['clean_text'].iloc[1][:120])

## Step 5: Build Vocabulary and Encode Text

⚠️ **This is the first major difference from the TensorFlow version.**

- **TensorFlow used:** `Keras Tokenizer` — a built-in class that does everything automatically
- **PyTorch uses:** We build the vocabulary manually using `collections.Counter`

The result is the same: each word gets a unique integer ID.


In [ ]:
def build_vocabulary(texts, max_vocab=MAX_VOCAB):
    """
    Builds a word-to-integer mapping from a list of cleaned comments.

    How it works:
    1. Count how many times every word appears across ALL comments
    2. Keep only the top max_vocab most common words
    3. Assign a unique integer ID to each word:
       - ID 0 = <PAD>  (used to fill short comments)
       - ID 1 = <OOV>  (used for unknown words at inference time)
       - ID 2+ = real words ordered by frequency

    Returns: vocab dictionary {word: integer_id}
    """
    word_counts = collections.Counter()

    for text in texts:
        word_counts.update(text.split())

    # Keep only the most frequent words
    most_common_words = word_counts.most_common(max_vocab)

    # Build the dictionary: PAD and OOV first, then real words
    vocab = {PAD_TOKEN: 0, OOV_TOKEN: 1}
    for idx, (word, count) in enumerate(most_common_words, start=2):
        vocab[word] = idx

    print(f'Total unique words found  : {len(word_counts):,}')
    print(f'Vocabulary size (kept)    : {len(vocab):,}  (includes PAD + OOV)')

    return vocab


# Build vocabulary ONLY from training data
# We never look at test data when building the vocabulary
vocab = build_vocabulary(train_df['clean_text'].tolist(), MAX_VOCAB)

# Show a few sample word IDs
sample_words = ['the', 'you', 'stupid', 'amazing', 'hate', '<PAD>', '<OOV>']
print('\nSample word IDs:')
for w in sample_words:
    print(f'  "{w}" → {vocab.get(w, "not in vocab")}')

In [ ]:
def encode_and_pad(texts, vocab, max_seq_len=MAX_SEQ_LEN):
    """
    Converts a list of cleaned comments into a 2D NumPy array.
    Shape: (number_of_comments, max_seq_len)

    For each comment:
    - Each word is replaced by its integer ID from vocab
    - Unknown words → OOV ID (1)
    - Comments longer than max_seq_len are trimmed
    - Comments shorter than max_seq_len are padded with 0s

    In TensorFlow this was done with:
        tokenizer.texts_to_sequences() + pad_sequences()
    In PyTorch we write it manually.
    """
    oov_id = vocab[OOV_TOKEN]   # ID for unknown words
    pad_id = vocab[PAD_TOKEN]   # ID for padding (always 0)
    encoded = []

    for text in texts:
        # Convert each word to its integer ID
        ids = [vocab.get(word, oov_id) for word in text.split()]
        # Trim if comment is too long
        ids = ids[:max_seq_len]
        # Pad with zeros if comment is too short
        ids = ids + [pad_id] * (max_seq_len - len(ids))
        encoded.append(ids)

    return np.array(encoded, dtype=np.int64)


# Encode all comments
print('Encoding and padding training comments...')
X_all_encoded = encode_and_pad(train_df['clean_text'].tolist(), vocab, MAX_SEQ_LEN)

print('Encoding and padding test comments...')
X_test_encoded = encode_and_pad(test_df['clean_text'].tolist(), vocab, MAX_SEQ_LEN)

print(f'\nX_all_encoded shape  : {X_all_encoded.shape}')   # (159571, 200)
print(f'X_test_encoded shape : {X_test_encoded.shape}')   # (153164, 200)

# Show what the first comment looks like after encoding
print('\nFirst comment encoded (first 20 values):')
print(X_all_encoded[0][:20])

In [ ]:
# Prepare labels
y_all = train_df[LABEL_COLS].values.astype(np.float32)
print('Labels shape:', y_all.shape)   # (159571, 6)

# Train / Validation split (80% train, 20% val)
X_train, X_val, y_train, y_val = train_test_split(
    X_all_encoded, y_all,
    test_size=VAL_SPLIT,
    random_state=42
)

print(f'Training samples   : {X_train.shape[0]:,}')
print(f'Validation samples : {X_val.shape[0]:,}')

## Step 5b: Create PyTorch Dataset and DataLoader

⚠️ **This step does not exist in the TensorFlow version.**

- **TensorFlow:** `model.fit()` accepts raw NumPy arrays directly — no extra work needed
- **PyTorch:** We must wrap our data in a `Dataset` class and a `DataLoader`

A `Dataset` teaches PyTorch how to access one sample.  
A `DataLoader` handles batching, shuffling, and feeding batches to the model.


In [ ]:
class ToxicCommentDataset(Dataset):
    """
    Custom PyTorch Dataset class.

    Every PyTorch Dataset must implement:
    - __len__     : how many samples are in the dataset?
    - __getitem__ : return one (input, label) pair as tensors

    The DataLoader will call __getitem__ many times and
    stack the results into batches automatically.
    """

    def __init__(self, X, y=None):
        # Convert NumPy arrays to PyTorch tensors
        self.X = torch.tensor(X, dtype=torch.long)           # word IDs must be long (int64)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]   # for test data (no labels)


# Create Dataset objects
train_dataset = ToxicCommentDataset(X_train, y_train)
val_dataset   = ToxicCommentDataset(X_val,   y_val)
test_dataset  = ToxicCommentDataset(X_test_encoded)

# Create DataLoaders
# shuffle=True for training so the model doesn't memorize the order
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=256,        shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=256,        shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

# Check one batch
sample_X, sample_y = next(iter(train_loader))
print(f'\nSample batch — X shape : {sample_X.shape}')  # (128, 200)
print(f'Sample batch — y shape : {sample_y.shape}')   # (128, 6)

## Step 6: Build the Bidirectional LSTM Model

⚠️ **This is the second major difference from the TensorFlow version.**

- **TensorFlow used:** `keras.Sequential([Layer1, Layer2, ...])`
- **PyTorch uses:** A Python class that inherits from `nn.Module`

In the class:
- `__init__` defines all the layers
- `forward` defines how data flows through those layers

The **architecture is identical** — same layers, same sizes, same logic.


In [ ]:
class BiLSTMClassifier(nn.Module):
    """
    Bidirectional LSTM model for multi-label toxicity classification.

    Architecture (same as TensorFlow version):
        Embedding → BiLSTM → Global Max Pool → Dense(relu) → Dropout → Dense(sigmoid)
    """

    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_labels, dropout=0.3):
        super(BiLSTMClassifier, self).__init__()

        # ── Layer 1: Embedding ────────────────────────────────────────────────
        # Converts each word ID (integer) into a dense vector of size embedding_dim
        # padding_idx=0 tells the model to ignore PAD tokens (they are always 0)
        # In Keras: Embedding(input_dim=vocab_size, output_dim=embedding_dim)
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # ── Layer 2: Bidirectional LSTM ───────────────────────────────────────
        # batch_first=True means input shape is (batch, seq_len, features)
        # bidirectional=True → reads text left→right AND right→left
        # output hidden size = hidden_dim * 2 (both directions concatenated)
        # In Keras: Bidirectional(LSTM(64, return_sequences=True))
        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # ── Layer 3: Dense (fully connected) ─────────────────────────────────
        # Input: hidden_dim * 2 because BiLSTM doubles the hidden size
        # In Keras: Dense(64, activation='relu')
        self.fc1 = nn.Linear(hidden_dim * 2, 64)

        # ── Layer 4: Dropout ──────────────────────────────────────────────────
        # Randomly zeroes 30% of neurons during training to prevent overfitting
        # In Keras: Dropout(0.3)
        self.dropout = nn.Dropout(dropout)

        # ── Layer 5: Output layer ─────────────────────────────────────────────
        # 64 → 6 outputs, one probability per toxicity label
        # In Keras: Dense(6, activation='sigmoid')
        self.fc2 = nn.Linear(64, num_labels)

        # Activation functions
        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()  # sigmoid gives 0–1 probability per label independently

    def forward(self, x):
        """
        Defines the forward pass — how input moves through each layer.
        In TensorFlow, Sequential() handles this automatically.
        In PyTorch, we write it ourselves.
        """
        # x shape: (batch_size, MAX_SEQ_LEN)

        # Step 1: Embedding
        embedded = self.embedding(x)
        # embedded shape: (batch_size, MAX_SEQ_LEN, embedding_dim)

        # Step 2: BiLSTM — returns output at every time step
        lstm_out, _ = self.bilstm(embedded)
        # lstm_out shape: (batch_size, MAX_SEQ_LEN, hidden_dim * 2)

        # Step 3: Global Max Pooling
        # In Keras: GlobalMaxPooling1D()
        # In PyTorch: torch.max() across the sequence dimension (dim=1)
        # This picks the most important value from each feature across all time steps
        pooled, _ = torch.max(lstm_out, dim=1)
        # pooled shape: (batch_size, hidden_dim * 2)

        # Step 4: Dense + ReLU
        out = self.relu(self.fc1(pooled))
        # out shape: (batch_size, 64)

        # Step 5: Dropout
        out = self.dropout(out)

        # Step 6: Output + Sigmoid
        out = self.sigmoid(self.fc2(out))
        # out shape: (batch_size, 6) — one probability per label

        return out


# Build the model and move it to the correct device (CPU or GPU)
model = BiLSTMClassifier(
    vocab_size    = len(vocab),
    embedding_dim = EMBEDDING_DIM,
    hidden_dim    = HIDDEN_DIM,
    num_labels    = len(LABEL_COLS),
    dropout       = 0.3
).to(DEVICE)

# Print model summary
print(model)

# Count total trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters : {total_params:,}')

## Step 7: Train the Model

⚠️ **This is the third major difference from the TensorFlow version.**

- **TensorFlow:** `model.fit(X_train, y_train, epochs=5, ...)` — one line handles everything
- **PyTorch:** We write the training loop ourselves — for every epoch, for every batch

This gives more control and is how professional PyTorch code is written.

Each iteration of the inner loop does 4 things:
1. **Forward pass** — get predictions
2. **Compute loss** — how wrong were we?
3. **Backward pass** — compute gradients (`loss.backward()`)
4. **Update weights** — improve the model (`optimizer.step()`)


In [ ]:
# ── Setup: optimizer and loss function ───────────────────────────────────────
# Adam optimizer — same as TensorFlow version
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# BCELoss = Binary Cross-Entropy Loss
# Same as 'binary_crossentropy' in Keras
# We use BCELoss (not BCEWithLogitsLoss) because our model already applies sigmoid
criterion = nn.BCELoss()

# ── History dictionary to store metrics per epoch ────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

# ── Early stopping variables (replaces Keras EarlyStopping callback) ─────────
best_val_loss    = float('inf')   # start with a very large number
patience         = 2              # stop if no improvement for 2 epochs
patience_counter = 0

print(f'Starting training on {DEVICE}...\n')
print(f'{"Epoch":>6}  {"Train Loss":>11}  {"Train Acc":>10}  {"Val Loss":>9}  {"Val Acc":>8}')
print('-' * 57)

for epoch in range(1, EPOCHS + 1):

    # ── TRAINING PHASE ───────────────────────────────────────────────────────
    # model.train() enables Dropout — always call this before training
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total   = 0

    for batch_X, batch_y in train_loader:

        # Move data to the same device as the model
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        # Step 1: Reset gradients
        # PyTorch accumulates gradients by default — we must clear them each batch
        optimizer.zero_grad()

        # Step 2: Forward pass — get predictions
        predictions = model(batch_X)

        # Step 3: Calculate loss
        loss = criterion(predictions, batch_y)

        # Step 4: Backward pass — compute how each weight contributed to the error
        loss.backward()

        # Step 5: Update weights based on gradients
        optimizer.step()

        # Track metrics
        train_loss    += loss.item() * batch_X.size(0)
        pred_labels    = (predictions >= THRESHOLD).float()
        train_correct += (pred_labels == batch_y).all(dim=1).sum().item()
        train_total   += batch_X.size(0)

    # ── VALIDATION PHASE ─────────────────────────────────────────────────────
    # model.eval() disables Dropout — always call this before validation/testing
    model.eval()
    val_loss    = 0.0
    val_correct = 0
    val_total   = 0

    # torch.no_grad() tells PyTorch not to track gradients — saves memory and speed
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            predictions = model(batch_X)
            loss        = criterion(predictions, batch_y)

            val_loss    += loss.item() * batch_X.size(0)
            pred_labels  = (predictions >= THRESHOLD).float()
            val_correct += (pred_labels == batch_y).all(dim=1).sum().item()
            val_total   += batch_X.size(0)

    # ── Calculate averages ────────────────────────────────────────────────────
    avg_train_loss = train_loss / train_total
    avg_val_loss   = val_loss   / val_total
    avg_train_acc  = train_correct / train_total
    avg_val_acc    = val_correct   / val_total

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_acc'].append(avg_train_acc)
    history['val_acc'].append(avg_val_acc)

    print(f'{epoch:>6}  {avg_train_loss:>11.4f}  {avg_train_acc:>10.4f}  '
          f'{avg_val_loss:>9.4f}  {avg_val_acc:>8.4f}')

    # ── Save best model (replaces Keras ModelCheckpoint callback) ─────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # torch.save() saves only the model WEIGHTS (state_dict)
        # In Keras: model.save('model.h5') saved the full model
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  ✅ Best model saved (val_loss improved to {best_val_loss:.4f})')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'\n  Early stopping after epoch {epoch} — no improvement for {patience} epochs')
            break

# Load the best weights back (same as restore_best_weights=True in Keras)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
print('\nTraining complete! Best model weights restored.')

## Step 8: Plot Training History


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss',      color='steelblue', marker='o')
axes[0].plot(history['val_loss'],   label='Validation Loss', color='tomato',    marker='s')
axes[0].set_title('Model Loss over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history['train_acc'], label='Train Accuracy',      color='steelblue', marker='o')
axes[1].plot(history['val_acc'],   label='Validation Accuracy', color='tomato',    marker='s')
axes[1].set_title('Model Accuracy over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()
print('Saved: training_history.png')

last = len(history['train_loss']) - 1
print(f'\nFinal train loss     : {history["train_loss"][last]:.4f}')
print(f'Final val loss       : {history["val_loss"][last]:.4f}')
print(f'Final train accuracy : {history["train_acc"][last]:.4f}')
print(f'Final val accuracy   : {history["val_acc"][last]:.4f}')

## Step 9: Evaluate the Model

⚠️ **PyTorch difference:** We cannot pass NumPy arrays directly to the model.
We must use a DataLoader and call `model(tensor)` inside `torch.no_grad()`.


In [ ]:
# Get predictions on the validation set
model.eval()   # IMPORTANT: disable dropout before predicting

all_probs = []

with torch.no_grad():   # No gradient calculation needed — saves memory
    for batch_X, batch_y in val_loader:
        batch_X = batch_X.to(DEVICE)
        probs   = model(batch_X)
        # .cpu() moves tensor from GPU back to CPU
        # .numpy() converts PyTorch tensor to NumPy array
        all_probs.append(probs.cpu().numpy())

# Stack all batches into a single array
y_pred_prob = np.vstack(all_probs)

# Convert probabilities to binary 0/1 using threshold
y_pred = (y_pred_prob >= THRESHOLD).astype(int)

print(f'Predictions shape : {y_pred_prob.shape}')  # (31914, 6)
print(f'y_val shape       : {y_val.shape}')         # (31914, 6)

In [ ]:
# Per-label metrics
print(f'{"Label":<16}  {"Precision":>9}  {"Recall":>7}  {"F1":>6}')
print('-' * 44)

auc_scores = []
for i, label in enumerate(LABEL_COLS):
    f1  = f1_score(y_val[:, i],       y_pred[:, i],      zero_division=0)
    pre = precision_score(y_val[:, i], y_pred[:, i],     zero_division=0)
    rec = recall_score(y_val[:, i],    y_pred[:, i],     zero_division=0)
    auc = roc_auc_score(y_val[:, i],   y_pred_prob[:, i])
    auc_scores.append(auc)
    print(f'{label:<16}  {pre:>9.3f}  {rec:>7.3f}  {f1:>6.3f}')

In [ ]:
# Overall metrics
h_loss  = hamming_loss(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_pred_prob, average='macro')

print(f'Hamming Loss    : {h_loss:.4f}   (lower is better, 0 = perfect)')
print(f'ROC-AUC (macro) : {roc_auc:.4f}   (higher is better, 1.0 = perfect)')

In [ ]:
# ROC-AUC bar chart — one bar per label
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6']

plt.figure(figsize=(9, 5))
bars = plt.bar(
    [l.replace('_', ' ').title() for l in LABEL_COLS],
    auc_scores, color=colors, edgecolor='black', alpha=0.85
)
plt.ylim(0, 1)
plt.title('ROC-AUC Score per Toxicity Label', fontsize=13, fontweight='bold')
plt.ylabel('ROC-AUC Score')
plt.xticks(rotation=20, ha='right')
for bar, score in zip(bars, auc_scores):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.01,
             f'{score:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('roc_auc_scores.png', dpi=150)
plt.show()
print('Saved: roc_auc_scores.png')

## Step 10: Test the Model on Sample Comments

Write a `predict_toxicity()` function that works on a single raw comment string.
This function is also used by the Streamlit app (`app_pytorch.py`).


In [ ]:
def predict_toxicity(comment, model, vocab, threshold=THRESHOLD):
    """
    Takes a raw comment string and returns toxicity probabilities.

    Steps:
    1. Clean the text
    2. Encode words to integers using vocab
    3. Pad to MAX_SEQ_LEN
    4. Convert to PyTorch tensor and move to DEVICE
    5. Run model in eval mode (no gradient, no dropout)
    6. Return dict {label: probability}
    """
    # Step 1: Clean
    cleaned = clean_text(comment)

    # Step 2 & 3: Encode and pad
    encoded = encode_and_pad([cleaned], vocab, MAX_SEQ_LEN)

    # Step 4: Convert to PyTorch tensor and move to device
    tensor = torch.tensor(encoded, dtype=torch.long).to(DEVICE)

    # Step 5: Model inference
    model.eval()
    with torch.no_grad():
        # model() calls forward() internally
        probs = model(tensor)[0].cpu().numpy()   # [0] to get the first (only) item

    # Step 6: Return as dictionary
    return {label: float(prob) for label, prob in zip(LABEL_COLS, probs)}


print('predict_toxicity() function defined!')

In [ ]:
# Test on sample comments
sample_comments = [
    "I love your work! This is really amazing and helpful.",
    "You are so stupid and worthless, nobody likes you!",
    "I will find you and make you regret this, just wait.",
    "Great article, very informative and well written.",
    "This is complete garbage written by an idiot.",
]

for comment in sample_comments:
    results = predict_toxicity(comment, model, vocab)
    display = comment[:70] + '...' if len(comment) > 70 else comment
    print(f'\nComment: "{display}"')
    print('-' * 55)
    for label, prob in results.items():
        flag = '🚨 DETECTED' if prob >= THRESHOLD else '✅ Clean'
        print(f'  {label:<16}: {prob:.3f}  {flag}')

## Step 11: Generate Predictions on the Test Set


In [ ]:
# Run model on all 153,164 test comments using the DataLoader
model.eval()
all_test_probs = []

print(f'Predicting on {len(test_df):,} test comments...')

with torch.no_grad():
    for (batch_X,) in test_loader:   # test_loader has no labels, just X
        batch_X = batch_X.to(DEVICE)
        probs   = model(batch_X)
        all_test_probs.append(probs.cpu().numpy())

# Combine all batches
test_preds = np.vstack(all_test_probs)

# Build submission DataFrame
submission_df = pd.DataFrame(test_preds, columns=LABEL_COLS)
submission_df.insert(0, 'id', test_df['id'])

# Save to CSV
submission_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')
submission_df.head()

## Step 12: Save Model, Vocabulary, and Config

⚠️ **PyTorch saving is different from TensorFlow.**

| | TensorFlow | PyTorch |
|---|---|---|
| Save model | `model.save('model.h5')` — saves everything | `torch.save(model.state_dict(), path)` — saves weights only |
| Save tokenizer | `pickle.dump(tokenizer, f)` | `pickle.dump(vocab, f)` — save vocabulary dict |
| Load model | `load_model('model.h5')` — just one line | Must recreate architecture first, then load weights |

We also save a `model_config_pytorch.pkl` file with the architecture settings
so the Streamlit app can rebuild the model without any hardcoded values.


In [ ]:
# 1. Save model weights
# torch.save() saves the state_dict — a dictionary of all weight tensors
torch.save(model.state_dict(), MODEL_PATH)
print(f'Model weights saved : {MODEL_PATH}')

# 2. Save vocabulary dictionary
with open(VOCAB_PATH, 'wb') as f:
    pickle.dump(vocab, f)
print(f'Vocabulary saved    : {VOCAB_PATH}')

# 3. Save model architecture config
# This is needed to rebuild the model for the Streamlit app
model_config = {
    'vocab_size'    : len(vocab),
    'embedding_dim' : EMBEDDING_DIM,
    'hidden_dim'    : HIDDEN_DIM,
    'num_labels'    : len(LABEL_COLS),
    'max_seq_len'   : MAX_SEQ_LEN,
}
with open(CONFIG_PATH, 'wb') as f:
    pickle.dump(model_config, f)
print(f'Model config saved  : {CONFIG_PATH}')

print('\nAll artifacts saved! Ready for Streamlit deployment.')
print('Run:  streamlit run app_pytorch.py')

In [ ]:
# ── How to RELOAD the model later (e.g. in app_pytorch.py) ──────────────────

# Step 1: Load config and vocab
# with open('model_config_pytorch.pkl', 'rb') as f:
#     config = pickle.load(f)
# with open('vocab_pytorch.pkl', 'rb') as f:
#     vocab = pickle.load(f)

# Step 2: Rebuild the architecture (must match exactly what was used in training)
# loaded_model = BiLSTMClassifier(
#     vocab_size    = config['vocab_size'],
#     embedding_dim = config['embedding_dim'],
#     hidden_dim    = config['hidden_dim'],
#     num_labels    = config['num_labels'],
# )

# Step 3: Load saved weights into the model
# loaded_model.load_state_dict(torch.load('toxicity_model_pytorch.pth', map_location=DEVICE))
# loaded_model.eval()

print('Reload pattern shown above (commented out — run when needed)')

## Step 13: Final Summary


In [ ]:
print('=' * 65)
print('   COMMENT TOXICITY DETECTION (PyTorch) — FINAL SUMMARY')
print('=' * 65)
print(f'Framework              : PyTorch {torch.__version__}')
print(f'Device used            : {DEVICE}')
print(f'Training samples       : 159,571')
print(f'Test samples           : 153,164')
print(f'Toxicity labels        : {len(LABEL_COLS)}  (multi-label classification)')
print(f'Vocabulary size        : {len(vocab):,} words')
print(f'Sequence length        : {MAX_SEQ_LEN} tokens')
print(f'Model architecture     : Bidirectional LSTM')
print(f'Total parameters       : ~{sum(p.numel() for p in model.parameters()):,}')
print(f'Epochs trained         : {len(history["train_loss"])}')
print(f'Hamming Loss           : {h_loss:.4f}')
print(f'ROC-AUC Score (macro)  : {roc_auc:.4f}')
print()
print('Files generated:')
print(f'  -> {MODEL_PATH}')
print(f'  -> {VOCAB_PATH}')
print(f'  -> {CONFIG_PATH}')
print(f'  -> {OUTPUT_CSV}')
print(f'  -> label_distribution.png')
print(f'  -> class_imbalance.png')
print(f'  -> comment_lengths.png')
print(f'  -> label_cooccurrence.png')
print(f'  -> training_history.png')
print(f'  -> roc_auc_scores.png')
print()
print('To launch the Streamlit web app, run:')
print('    streamlit run app_pytorch.py')
print('=' * 65)